# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click !here goes the icon of the corresponding button in the gutter! button.
To debug a cell, press Alt+Shift+Enter, or click !here goes the icon of the corresponding button in the gutter! button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/jupyter-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [4]:
# =========================
# 配置
# =========================
X_PATH = "X_gridblocks_flat.csv"
YB_PATH = "Y_build.csv"
YL_PATH = "Y_level.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
BATCH_SIZE = 64
EPOCHS = 500
LR = 1e-3
WEIGHT_DECAY = 1e-4

N_POINTS = 100
FEAT_DIM = 49
IN_DIM = N_POINTS * FEAT_DIM  # 4900

# 10/25/40 对应等级 1/2/3（0 表示不建站）
LEVEL_TO_PILES = {0: 0, 1: 10, 2: 25, 3: 40}

# Transformer/Set-Encoder 超参数
D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 2
FF_DIM = 256
DROPOUT = 0.10

# =========================
# 工具函数
# =========================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_csv_float(path):
    return np.loadtxt(path, delimiter=",", dtype=np.float32)

def load_csv_int(path):
    return np.loadtxt(path, delimiter=",", dtype=np.int64)

class StandardScaler:
    """只用训练集拟合，避免泄露"""
    def __init__(self, eps=1e-8):
        self.mean = None
        self.std = None
        self.eps = eps

    def fit(self, x: np.ndarray):
        self.mean = x.mean(axis=0, keepdims=True)
        self.std = x.std(axis=0, keepdims=True)
        self.std = np.maximum(self.std, self.eps)

    def transform(self, x: np.ndarray):
        return (x - self.mean) / self.std

    def fit_transform(self, x: np.ndarray):
        self.fit(x)
        return self.transform(x)

def compute_pad_mask_from_raw_X(X_raw_flat: np.ndarray) -> np.ndarray:
    """
    用原始 X（标准化前）推断 padding 点。
    约定：padding 点的 49 维特征全 0。
    返回: pad_mask [N,100]，True 表示该点是 padding。
    """
    X3 = X_raw_flat.reshape(-1, N_POINTS, FEAT_DIM)
    pad_mask = np.all(X3 == 0.0, axis=-1)
    return pad_mask

class BlocksDataset(Dataset):
    """
    X: [N, 4900] -> 每次取样 reshape 成 [100,49]
    Y_build: [N, 100] (0/1)
    Y_level: [N, 100] (0/1/2/3)
    pad_mask: [N,100] (True=padding)
    """
    def __init__(self, X, Yb, Yl, pad_mask):
        self.X = torch.from_numpy(X).float()
        self.Yb = torch.from_numpy(Yb).float()
        self.Yl = torch.from_numpy(Yl).long()
        self.pad_mask = torch.from_numpy(pad_mask).bool()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].reshape(N_POINTS, FEAT_DIM)  # [100,49]
        return x, self.Yb[idx], self.Yl[idx], self.pad_mask[idx]

# =========================
# 模型：Set-Encoder(Transformer) + 两个点级 head
# =========================
class SetTransformerHeads(nn.Module):
    """
    输入:  x [B,100,49]
    输出:  build_logits [B,100]
          level_logits [B,100,4]
    """
    def __init__(
        self,
        feat_dim: int,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(feat_dim, d_model),
            nn.LayerNorm(d_model),
        )

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.build_head = nn.Linear(d_model, 1)   # -> [B,100,1]
        self.level_head = nn.Linear(d_model, 4)   # -> [B,100,4]

        nn.init.zeros_(self.build_head.bias)
        nn.init.zeros_(self.level_head.bias)

    def forward(self, x, key_padding_mask=None):
        """
        x: [B,100,49]
        key_padding_mask: [B,100]  True=padding，会被 attention 忽略
        """
        h = self.input_proj(x)  # [B,100,d]
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)  # [B,100,d]

        build_logits = self.build_head(h).squeeze(-1)  # [B,100]
        level_logits = self.level_head(h)              # [B,100,4]
        return build_logits, level_logits

def masked_losses_and_metrics(build_logits, level_logits, Yb, Yl, pad_mask, bce, ce, level_w=0.7):
    """
    只在非 padding 点上计算 loss/acc，避免 padding 把指标“抬高”。
    pad_mask: True=padding
    """
    valid = ~pad_mask  # [B,100]
    valid_count = valid.sum().item()

    if valid_count == 0:
        loss = build_logits.sum() * 0.0
        return loss, 0.0, 0.0, 0

    # ----- loss -----
    build_logits_v = build_logits[valid].reshape(-1)
    Yb_v = Yb[valid].reshape(-1)
    loss_build = bce(build_logits_v, Yb_v)

    level_logits_v = level_logits[valid]   # [Nv,4]
    Yl_v = Yl[valid].reshape(-1)           # [Nv]
    loss_level = ce(level_logits_v, Yl_v)

    loss = loss_build + level_w * loss_level

    # ----- acc -----
    build_pred_v = (torch.sigmoid(build_logits_v) > 0.5).float()
    build_acc = (build_pred_v == Yb_v).float().mean().item()

    lvl_pred_v = level_logits_v.argmax(dim=-1)
    level_acc = (lvl_pred_v == Yl_v).float().mean().item()

    return loss, build_acc, level_acc, valid_count

# =========================
# 评估：返回 loss + build_acc + level_acc（只统计非 padding 点）
# =========================
@torch.no_grad()
def eval_model(model, loader, bce, ce, level_w=0.7):
    model.eval()

    loss_sum = 0.0
    valid_sum = 0

    build_acc_sum = 0.0
    level_acc_sum = 0.0
    n_batches = 0

    for X, Yb, Yl, pad_mask in loader:
        X = X.to(DEVICE)
        Yb = Yb.to(DEVICE)
        Yl = Yl.to(DEVICE)
        pad_mask = pad_mask.to(DEVICE)

        build_logits, level_logits = model(X, key_padding_mask=pad_mask)

        loss, build_acc, level_acc, valid_count = masked_losses_and_metrics(
            build_logits, level_logits, Yb, Yl, pad_mask, bce, ce, level_w=level_w
        )

        loss_sum += loss.item() * max(valid_count, 1)
        valid_sum += max(valid_count, 1)

        build_acc_sum += build_acc
        level_acc_sum += level_acc
        n_batches += 1

    avg_loss = loss_sum / max(valid_sum, 1)
    build_acc = build_acc_sum / max(n_batches, 1)
    level_acc = level_acc_sum / max(n_batches, 1)
    return avg_loss, build_acc, level_acc

def main():
    set_seed(SEED)

    assert os.path.isfile(X_PATH), f"找不到 {X_PATH}"
    assert os.path.isfile(YB_PATH), f"找不到 {YB_PATH}"
    assert os.path.isfile(YL_PATH), f"找不到 {YL_PATH}"

    X_raw = load_csv_float(X_PATH)   # [N,4900]
    Yb = load_csv_int(YB_PATH)       # [N,100]
    Yl = load_csv_int(YL_PATH)       # [N,100]

    if X_raw.ndim != 2 or X_raw.shape[1] != IN_DIM:
        raise ValueError(f"X 维度应为 [N,{IN_DIM}]，实际 {X_raw.shape}")

    if Yb.shape[0] != X_raw.shape[0] or Yb.shape[1] != N_POINTS:
        raise ValueError(f"Y_build 应为 [N,{N_POINTS}]，实际 {Yb.shape}")
    if Yl.shape[0] != X_raw.shape[0] or Yl.shape[1] != N_POINTS:
        raise ValueError(f"Y_level 应为 [N,{N_POINTS}]，实际 {Yl.shape}")

    # padding mask（必须用标准化前的 raw X 推断）
    pad_mask_all = compute_pad_mask_from_raw_X(X_raw)  # [N,100]

    N = X_raw.shape[0]
    print(f"Loaded: X={X_raw.shape}, Yb={Yb.shape}, Yl={Yl.shape}")

    # ========= 数据切分 =========
    idx = np.arange(N)
    np.random.shuffle(idx)

    n_train = int(0.7 * N)
    n_val = int(0.15 * N)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]

    # ========= 标准化（只用训练集拟合） =========
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_raw[train_idx])
    X_val = scaler.transform(X_raw[val_idx])
    X_test = scaler.transform(X_raw[test_idx])

    Yb_train, Yl_train = Yb[train_idx], Yl[train_idx]
    Yb_val, Yl_val = Yb[val_idx], Yl[val_idx]
    Yb_test, Yl_test = Yb[test_idx], Yl[test_idx]

    pad_train = pad_mask_all[train_idx]
    pad_val = pad_mask_all[val_idx]
    pad_test = pad_mask_all[test_idx]

    # ========= 类别权重（只统计非 padding 点） =========
    valid_train = ~pad_train  # [Ntrain,100]

    # build 正类权重 pos_weight = neg/pos
    yb_valid = Yb_train[valid_train]
    pos = yb_valid.sum()
    neg = yb_valid.size - pos
    pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=DEVICE)

    # level 的权重：只统计非 padding 点（含0类）
    yl_valid = Yl_train[valid_train].reshape(-1)
    counts = np.bincount(yl_valid, minlength=4).astype(np.float32) + 1.0
    inv = 1.0 / counts
    lvl_weight = torch.tensor(inv / inv.sum() * 4.0, dtype=torch.float32, device=DEVICE)

    print(f"pos_weight(build) = {pos_weight.item():.3f}")
    print(f"level_class_counts (valid only) = {counts}")
    print(f"level_class_weight = {lvl_weight.detach().cpu().numpy()}")

    # ========= DataLoader =========
    train_ds = BlocksDataset(X_train, Yb_train, Yl_train, pad_train)
    val_ds = BlocksDataset(X_val, Yb_val, Yl_val, pad_val)
    test_ds = BlocksDataset(X_test, Yb_test, Yl_test, pad_test)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # ========= Model =========
    model = SetTransformerHeads(
        feat_dim=FEAT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_layers=NUM_LAYERS,
        dim_feedforward=FF_DIM,
        dropout=DROPOUT,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # 损失
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    ce = nn.CrossEntropyLoss(weight=lvl_weight)

    best_val = -1.0
    best_state = None

    # ========= Train =========
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        valid_sum = 0

        for Xb, Yb_batch, Yl_batch, pad_mask in train_loader:
            Xb = Xb.to(DEVICE)                 # [B,100,49]
            Yb_batch = Yb_batch.to(DEVICE)     # [B,100]
            Yl_batch = Yl_batch.to(DEVICE)     # [B,100]
            pad_mask = pad_mask.to(DEVICE)     # [B,100]

            build_logits, level_logits = model(Xb, key_padding_mask=pad_mask)

            loss, _, _, valid_count = masked_losses_and_metrics(
                build_logits, level_logits, Yb_batch, Yl_batch, pad_mask, bce, ce, level_w=0.7
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()

            loss_sum += loss.item() * max(valid_count, 1)
            valid_sum += max(valid_count, 1)

        train_loss = loss_sum / max(valid_sum, 1)

        # ===== 训练集/验证集：统一 eval 逻辑（只统计非 padding 点）=====
        train_eval_loss, train_build_acc, train_level_acc = eval_model(
            model, train_loader, bce, ce, level_w=0.7
        )
        val_loss, val_build_acc, val_level_acc = eval_model(
            model, val_loader, bce, ce, level_w=0.7
        )

        # 选最好模型（验证集 build/level 平均）
        score = (val_build_acc + val_level_acc) / 2.0
        if score > best_val:
            best_val = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d} | "
                f"train_loss={train_eval_loss:.4f} | train_build_acc={train_build_acc:.4f} | train_level_acc={train_level_acc:.4f} | "
                f"val_loss={val_loss:.4f} | val_build_acc={val_build_acc:.4f} | val_level_acc={val_level_acc:.4f}"
            )

    # ========= Test =========
    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_build_acc, test_level_acc = eval_model(model, test_loader, bce, ce, level_w=0.7)

    print("\n==== BEST MODEL TRAIN ====")
    best_train_loss, best_train_build_acc, best_train_level_acc = eval_model(model, train_loader, bce, ce, level_w=0.7)
    print(f"train_loss={best_train_loss:.4f}")
    print(f"train_build_acc={best_train_build_acc:.4f}")
    print(f"train_level_acc={best_train_level_acc:.4f}")

    print("\n==== BEST MODEL TEST ====")
    print(f"test_loss={test_loss:.4f}")
    print(f"test_build_acc={test_build_acc:.4f}")
    print(f"test_level_acc={test_level_acc:.4f}")

    # 保存模型（包含 scaler 的均值方差，方便推理）
    ckpt = {
        "model": model.state_dict(),
        "scaler_mean": scaler.mean,
        "scaler_std": scaler.std,
        "level_to_piles": LEVEL_TO_PILES,
        "n_points": N_POINTS,
        "feat_dim": FEAT_DIM,
        "arch": {
            "name": "SetTransformerHeads",
            "d_model": D_MODEL,
            "nhead": NHEAD,
            "num_layers": NUM_LAYERS,
            "ff_dim": FF_DIM,
            "dropout": DROPOUT,
        },
    }
    torch.save(ckpt, "set_transformer_blocks_ckpt.pt")
    print("Saved: set_transformer_blocks_ckpt.pt")

if __name__ == "__main__":
    main()



Loaded: X=(4054, 4900), Yb=(4054, 100), Yl=(4054, 100)
pos_weight(build) = 0.037
level_class_counts (valid only) = [ 10116. 129997. 100802.  42789.]
level_class_weight = [2.827678   0.22004193 0.28377205 0.66850805]


D:\Anaconda\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 001 | train_loss=0.1733 | train_build_acc=0.9297 | train_level_acc=0.8732 | val_loss=0.1682 | val_build_acc=0.9273 | val_level_acc=0.8732
Epoch 010 | train_loss=0.1218 | train_build_acc=0.9600 | train_level_acc=0.9104 | val_loss=0.1267 | val_build_acc=0.9589 | val_level_acc=0.9109
Epoch 020 | train_loss=0.1016 | train_build_acc=0.9661 | train_level_acc=0.9260 | val_loss=0.1192 | val_build_acc=0.9638 | val_level_acc=0.9251
Epoch 030 | train_loss=0.0928 | train_build_acc=0.9743 | train_level_acc=0.9301 | val_loss=0.1267 | val_build_acc=0.9712 | val_level_acc=0.9279
Epoch 040 | train_loss=0.0971 | train_build_acc=0.9745 | train_level_acc=0.9324 | val_loss=0.1637 | val_build_acc=0.9713 | val_level_acc=0.9285
Epoch 050 | train_loss=0.0838 | train_build_acc=0.9792 | train_level_acc=0.9393 | val_loss=0.1519 | val_build_acc=0.9744 | val_level_acc=0.9334
Epoch 060 | train_loss=0.0774 | train_build_acc=0.9807 | train_level_acc=0.9417 | val_loss=0.1377 | val_build_acc=0.9759 | val_level_acc